# Feature Engineering

### NFL Big Data Bowl 2027

by Chelle Davies

In [ ]:
%pip install pandas numpy matplotlib seaborn plotly scipy

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display
import scipy
import scipy.stats as stats

In [ ]:
# Load all CSV files in the "nfl-big-data-bowl-2027-data" directory into a dictionary of DataFrames

file_path = "nfl-big-data-bowl-2027-data"

dataframes = {
    os.path.splitext(filename)[0]: pd.read_csv(os.path.join(file_path, filename))
    for filename in os.listdir(file_path)
    if filename.lower().endswith(".csv")
}

list(dataframes)

In [ ]:
for name, df in dataframes.items():
    print(f"DataFrame: {name}, Shape: {df.shape}")
    print(df.info())
    print("\n")

---

#### Hypothesis 1: Higher mean combine acceleration is associated with more sacks per game.

#### Hypothesis 2: Greater combine acceleration variability is associated with more sacks per game.

Pass rush success relies on two distinct traits: explosive burst (mean capacity) and deceptive pacing/sudden redirection (variability).

- `top_decile_mean_accel`: The mean of the top 10% highest acceleration frames. Isolates the player's peak capacity rather than diluting it with baseline tracking frames.
- `accel_standard_deviation`: The standard deviation of acceleration across a complete drill. Directly tests the variability hypothesis, measuring a player's ability to violently modulate speed.
- `accel_coefficient_of_variation`: Calculated as `accel_standard_deviation` / `overall_mean_accel`. This normalizes variability against absolute speed, preventing elite athletes from skewing the variance metric.
- `jerk_variance`: The variance of the first derivative of acceleration. Captures how erratic or sudden the player's force adjustments are, representing a non-linear, unpredictable pass rush.


In [ ]:
combine_tracking_df = dataframes.get("combine_tracking")

In [ ]:
accel_col = "a" if "a" in combine_tracking_df.columns else "acceleration"

def top_decile_mean(values):
    values = values.dropna()
    if values.empty:
        return np.nan
    n = max(1, int(np.ceil(len(values) * 0.10)))
    return values.nlargest(n).mean()

combine_tracking_df["top_decile_mean_accel"] = (
    combine_tracking_df.groupby("nfl_id")[accel_col]
    .transform(top_decile_mean)
)

In [ ]:
tracking = combine_tracking_df.copy().reset_index(drop=True)
group_cols = ["nfl_id"] + (["event_id"] if "event_id" in tracking.columns else [])

In [ ]:
# Per-player acceleration metrics
player_accel = tracking.groupby("nfl_id")[accel_col]
accel_std = player_accel.transform("std")
accel_mean = player_accel.transform("mean")

combine_tracking_df["accel_standard_deviation"] = accel_std.to_numpy()
combine_tracking_df["accel_coefficient_of_variation"] = (
    accel_std / accel_mean.replace(0, np.nan)
).to_numpy()

In [ ]:
# Calculate jerk (change in acceleration per second) within each event
if "time" in tracking.columns:
    elapsed = pd.to_numeric(tracking["time"], errors="coerce")
else:
    elapsed = pd.Series(np.nan, index=tracking.index)

if elapsed.isna().all() and "frame_id" in tracking.columns:
    elapsed = pd.to_numeric(tracking["frame_id"], errors="coerce") / 10

tracking["_elapsed"] = elapsed
tracking["_accel"] = pd.to_numeric(tracking[accel_col], errors="coerce")
tracking = tracking.sort_values(group_cols + ["_elapsed"])

grouped = tracking.groupby(group_cols, sort=False)
dt = grouped["_elapsed"].diff()
da = grouped["_accel"].diff()
tracking["_jerk"] = (da / dt).where(dt > 0)

jerk_variance = tracking.groupby("nfl_id")["_jerk"].var()
combine_tracking_df["jerk_variance"] = (
    combine_tracking_df["nfl_id"].map(jerk_variance).to_numpy()
)

In [ ]:
combine_tracking_df.describe()

#### Hypothesis 3: Higher mean combine acceleration is associated with more receiving yards per game.

For receivers and tight ends, mean acceleration translates to vertical separation and explosive speed out of breaks.

- `break_window_mean_accel`: The mean acceleration in the 1.0-second window immediately following a major drop in velocity (the top of a route break).
- `linear_strides_mean_accel`: The mean acceleration during pure linear phases (e.g., frames 10 to 30 of the 40-yard dash). Measures sustained acceleration downfield.
- `accel_efficiency_ratio`: The ratio of forward-vector acceleration (`a_{x}`) to lateral acceleration (`a_{y}`). Higher ratios mean less wasted energy during downfield acceleration.


In [ ]:
# Engineer event-level movement features and repeat each value across its event rows.
features_df = combine_tracking_df.copy()
features_df["_time"] = pd.to_datetime(features_df["time"], errors="coerce", utc=True)

event_cols = ["nfl_id", "event_id"]
if "attempt" in features_df.columns:
    event_cols.append("attempt")

for col in [
    "break_window_mean_accel",
    "linear_strides_mean_accel",
    "accel_efficiency_ratio",
]:
    combine_tracking_df[col] = np.nan

In [ ]:
for _, event in features_df.groupby(event_cols, dropna=False, sort=False):
    event = event.loc[event["entity_type"].eq("PLAYER")].copy()
    event = event.dropna(subset=["_time"]).sort_values("_time")
    event = event.drop_duplicates("_time")

    if event.empty:
        continue

    idx = event.index
    drill_type = str(event["drill_type"].iloc[0]).upper()
    drill_name = str(event["drill_name"].iloc[0]).upper()
    elapsed = (event["_time"] - event["_time"].iloc[0]).dt.total_seconds().to_numpy()
    accel = pd.to_numeric(event[accel_col], errors="coerce").to_numpy()

    # Mean acceleration during the 1-second window after the strongest
    # observed speed decrease in a WR/TE drill.
    if ("SKILL_DRILLS_WR" in drill_type or "SKILL_DRILLS_TE" in drill_type):
        speed = pd.to_numeric(event["s"], errors="coerce").to_numpy()
        if len(event) >= 2:
            dt = np.diff(elapsed)
            ds = np.diff(speed)
            decel = np.divide(ds, dt, out=np.full_like(ds, np.nan), where=dt > 0)
            if np.isfinite(decel).any():
                onset = elapsed[np.nanargmin(decel) + 1]
                window = (elapsed >= onset) & (elapsed <= onset + 1.0)
                values = accel[window]
                if np.isfinite(values).any():
                    combine_tracking_df.loc[idx, "break_window_mean_accel"] = np.nanmean(values)

    # Frames 10–30 (inclusive) of the 40-yard dash.
    if "FORTY" in drill_name and len(event) >= 31:
        values = accel[10:31]
        if np.isfinite(values).any():
            combine_tracking_df.loc[idx, "linear_strides_mean_accel"] = np.nanmean(values)

    # Ratio of longitudinal to lateral acceleration, derived from x/y
    # positions and projected relative to the direction of travel.
    if len(event) >= 3:
        x = pd.to_numeric(event["x"], errors="coerce").to_numpy()
        y = pd.to_numeric(event["y"], errors="coerce").to_numpy()
        if np.isfinite(x).all() and np.isfinite(y).all() and np.all(np.diff(elapsed) > 0):
            vx = np.gradient(x, elapsed)
            vy = np.gradient(y, elapsed)
            ax = np.gradient(vx, elapsed)
            ay = np.gradient(vy, elapsed)

            speed_xy = np.hypot(vx, vy)
            moving = speed_xy > 1e-6
            if moving.any():
                longitudinal = np.abs((ax * vx + ay * vy) / speed_xy)
                lateral = np.abs((vx * ay - vy * ax) / speed_xy)
                lateral_mean = np.nanmean(lateral[moving])
                if np.isfinite(lateral_mean) and lateral_mean > 1e-8:
                    ratio = np.nanmean(longitudinal[moving]) / lateral_mean
                    combine_tracking_df.loc[idx, "accel_efficiency_ratio"] = ratio

In [ ]:
combine_tracking_df[
    ["break_window_mean_accel", "linear_strides_mean_accel", "accel_efficiency_ratio"]
].describe()

#### Hypothesis 4: Combine movement metrics improve held-out-year predictions beyond position and traditional combine results.

The aim here is to prevent leakage and isolate incremental feature importance.

- The Baseline Feature Group: `position_encoded` (one-hot), `official_40_time`, `official_3_cone_time`, `official_shuttle_time`, `height`, `weight`.
- The Tracking Feature Group: The engineered sensor metrics detailed above (`top_decile_mean_accel`, `accel_coefficient_of_variation`, etc.).

In [ ]:
combine_results_df = dataframes["combine_results"].copy()

# Use one combine-results row per player for the lookup.
player_lookup = combine_results_df.drop_duplicates("nfl_id", keep="first").set_index("nfl_id")

# Map combine-results fields to the requested feature names.
source_candidates = {
    "official_40_time": ["combine_40_time", "official_40_time", "forty_time", "forty"],
    "official_3_cone_time": ["combine_3cone", "combine_3_cone_time", "official_3_cone_time", "three_cone_time", "three_cone"],
    "official_shuttle_time": ["combine_shuttle", "combine_shuttle_time", "official_shuttle_time", "shuttle_time", "shuttle", "short_shuttle_time", "short_shuttle"],
    "height": ["combine_height", "height"],
    "weight": ["combine_weight", "weight"],
    "top_decile_mean_accel": ["top_decile_mean_accel"],
    "accel_coefficient_of_variation": ["accel_coefficient_of_variation"],
}

In [ ]:
for output_col, candidates in source_candidates.items():
    source_col = next((c for c in candidates if c in player_lookup.columns), None)

    if source_col is not None and source_col in combine_tracking_df.columns:
        # Keep existing tracking-derived values when the source is the tracking dataframe.
        continue
    elif source_col is not None:
        combine_tracking_df[output_col] = combine_tracking_df["nfl_id"].map(player_lookup[source_col])
    elif output_col in combine_tracking_df.columns:
        # Already present in the tracking data; leave it unchanged.
        pass
    else:
        combine_tracking_df[output_col] = np.nan

In [ ]:
# One-hot encode combine position and map indicators onto tracking rows.
position_col = next(
    (c for c in ["combine_position", "position"] if c in player_lookup.columns),
    None,
)

if position_col is not None:
    position_dummies = pd.get_dummies(
        player_lookup[position_col].fillna("UNKNOWN").astype(str).str.upper(),
        prefix="position_encoded",
        dtype=int,
    )
    for dummy_col in position_dummies.columns:
        combine_tracking_df[dummy_col] = (
            combine_tracking_df["nfl_id"].map(position_dummies[dummy_col]).fillna(0).astype(int)
        )

In [ ]:
combine_tracking_df[
    [
        "official_40_time",
        "official_3_cone_time",
        "official_shuttle_time",
        "height",
        "weight",
        "top_decile_mean_accel",
        "accel_coefficient_of_variation",
    ]
].head()

______

#### Framework for Model Evaluation

| Hypothesis | Methodology | Strategy |
| --- | --- | --- |
| Hypothesis 1 | Feature-Level Significance | Use a tree-based ensemble (e.g., LightGBM or XGBoost) to extract SHAP (SHapley Additive exPlanations) values. Check if accel_coefficient_of_variation and top_decile_mean_accel show significant, positive SHAP main effects against their respective targets. |
| Hypothesis 2 | Feature-Level Significance | Use a tree-based ensemble (e.g., LightGBM or XGBoost) to extract SHAP (SHapley Additive exPlanations) values. Check if accel_coefficient_of_variation and top_decile_mean_accel show significant, positive SHAP main effects against their respective targets. |
| Hypothesis 3 | Feature-Level Significance | Use a tree-based ensemble (e.g., LightGBM or XGBoost) to extract SHAP (SHapley Additive exPlanations) values. Check if accel_coefficient_of_variation and top_decile_mean_accel show significant, positive SHAP main effects against their respective targets. |
| Hypothesis 4 | Model-Level Incremental Lift | Root Mean Squared Error (RMSE) and R² (Delta R² [the full model's R² minus the baseline model's R²] must be greater than 0.) |

##### Steps:
1. Train Baseline Model (Position + Traditional Metrics)
2. Train Full Model (Baseline + 10 Hz Engineered Tracking Features)
3. Evaluate Incremental Lift on a Held-Out Year

In [ ]:
# Save the engineered tracking and results dataframes to CSV files for further analysis or modeling.

combine_tracking_df.to_csv(
    os.path.join(file_path, "combine_tracking_engineered.csv"),
    index=False,
)
combine_results_df.to_csv(
    os.path.join(file_path, "combine_results_copy.csv"),
    index=False,
)